# CoherenceBench-IN — IEEE Format (IEEEtran)

This notebook generates `paper/ieee/main_ieee.tex` ready for submission to an IEEE conference (2-column, `IEEEtran` document class).

**Authors:**
- Sowmya B J — AI & DS, Ramaiah Institute of Technology, `sowmyabj@msrit.edu`
- Jeeth Bhavesh Kataria — AI & DS, Ramaiah Institute of Technology, `jeethkataria9798@gmail.com`

**Run all cells** to regenerate after any section edits.

In [ ]:
# ─── Setup ────────────────────────────────────────────────────────
import json, pathlib, numpy as np, pandas as pd

BASE        = pathlib.Path('../')
RESULTS_DIR = BASE / 'results'
PAPER_DIR   = BASE / 'paper'
IEEE_DIR    = PAPER_DIR / 'ieee'
IEEE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILES = {
    'Llama-3.2-3B':  'llama3_3b',
    'Qwen2.5-3B':    'qwen25_3b',
    'Mistral-7B':    'mistral_7b',
    'Qwen2.5-7B':    'qwen25_7b',
    'Qwen2.5-14B':   'qwen25_14b',
}
DIMS    = ['entity_consistency', 'temporal_coherence', 'causal_chain']
DIM_LBL = ['Entity', 'Temporal', 'Causal']

results = {}
for name, fname in MODEL_FILES.items():
    fpath = RESULTS_DIR / f'{fname}_results.jsonl'
    if fpath.exists():
        results[name] = pd.DataFrame([json.loads(l) for l in open(fpath)])

def acc(df, **filt):
    s = df
    for c, v in filt.items():
        s = s[s[c] == v]
    return s['correct'].mean() if len(s) else float('nan')

def pct(v): return f'{v*100:.1f}\\%' if not np.isnan(v) else '---'

abstract_txt = (PAPER_DIR / 'abstract.txt').read_text() if (PAPER_DIR / 'abstract.txt').exists() else '% TODO: run 04_paper_writing.ipynb first'
print(f'✅ Setup complete. Loaded {len(results)} models. IEEE output → paper/ieee/')

In [ ]:
# ─── Table I — Main Results ───────────────────────────────────────
table_i = r"""\begin{table}[t]
\centering
\caption{Accuracy (\%) by Model and Coherence Dimension}
\label{tab:main}
\begin{tabular}{lcccc}
\hline
\textbf{Model} & \textbf{Entity} & \textbf{Temporal} & \textbf{Causal} & \textbf{Overall} \\
\hline
Random baseline & 50.0 & 50.0 & 50.0 & 50.0 \\
\hline
"""
for model, df in results.items():
    row = ' & '.join([model] + [f"{acc(df, dimension=d)*100:.1f}" for d in DIMS] + [f"{acc(df)*100:.1f}"])
    table_i += row + ' \\\\\n'
table_i += r"""\hline
\end{tabular}
\end{table}
"""

(IEEE_DIR / 'table1_main.tex').write_text(table_i)
print(table_i)

In [ ]:
# ─── Table II — Distance Breakdown ───────────────────────────────
table_ii = r"""\begin{table}[t]
\centering
\caption{INCONSISTENT-Instance Accuracy (\%) by Corruption Distance}
\label{tab:distance}
\begin{tabular}{lccc}
\hline
\textbf{Model} & \textbf{Near} & \textbf{Mid} & \textbf{Far} \\
\hline
"""
for model, df in results.items():
    inc = df[df['gold'] == 'INCONSISTENT']
    near = f"{acc(inc, distance='near')*100:.1f}"
    mid  = f"{acc(inc, distance='mid')*100:.1f}"
    far  = f"{acc(inc, distance='far')*100:.1f}"
    table_ii += f'{model} & {near} & {mid} & {far} \\\\\n'
table_ii += r"""\hline
\end{tabular}
\end{table}
"""
(IEEE_DIR / 'table2_distance.tex').write_text(table_ii)
print(table_ii)

In [ ]:
# ─── Copy shared bib ─────────────────────────────────────────────
import shutil
src_bib = PAPER_DIR / 'references.bib'
if src_bib.exists():
    shutil.copy(src_bib, IEEE_DIR / 'references.bib')
    print('Copied references.bib to paper/ieee/')
else:
    print('⚠️  Run 04_paper_writing.ipynb to generate references.bib first.')

In [ ]:
# ─── Assemble main_ieee.tex ───────────────────────────────────────

# Load section files generated by 04_paper_writing.ipynb
def load_sec(fname):
    f = PAPER_DIR / fname
    return f.read_text() if f.exists() else f'% TODO: run 04_paper_writing.ipynb to generate {fname}'

ieee_tex = r"""\documentclass[conference]{IEEEtran}
\IEEEoverridecommandlockouts
\usepackage{cite}
\usepackage{amsmath}
\usepackage{amssymb}
\usepackage{algorithmic}
\usepackage{graphicx}
\usepackage{textcomp}
\usepackage{xcolor}
\usepackage{booktabs}
\usepackage{microtype}
\usepackage{url}
\def\BibTeX{{\rm B\kern-.05em{\sc i\kern-.025em b}\kern-.08em T\kern-.1667em\lower.7ex\hbox{E}\kern-.125emX}}

\begin{document}

\title{CoherenceBench-IN: A Long-Context Discourse Coherence Benchmark\\
for Evaluating Large Language Models}

\author{
  \IEEEauthorblockN{Sowmya B J}
  \IEEEauthorblockA{\textit{Dept. of Artificial Intelligence and Data Science} \\
  \textit{Ramaiah Institute of Technology} \\
  Bangalore, India \\
  sowmyabj@msrit.edu}
  \and
  \IEEEauthorblockN{Jeeth Bhavesh Kataria}
  \IEEEauthorblockA{\textit{Dept. of Artificial Intelligence and Data Science} \\
  \textit{Ramaiah Institute of Technology} \\
  Bangalore, India \\
  jeethkataria9798@gmail.com}
}

\maketitle

\begin{abstract}
""" + abstract_txt + r"""
\end{abstract}

\begin{IEEEkeywords}
discourse coherence, large language models, long-context evaluation,
entity consistency, temporal coherence, causal reasoning, benchmark
\end{IEEEkeywords}

\section{Introduction}
""" + load_sec('sec_introduction.tex') + r"""

\section{Related Work}
""" + load_sec('sec_related_work.tex') + r"""

\section{CoherenceBench-IN}
""" + load_sec('sec_benchmark.tex') + r"""

\section{Experimental Setup}
""" + load_sec('sec_experiments.tex') + r"""

\section{Results}
""" + load_sec('sec_results.tex') + r"""

\section{Analysis}
""" + load_sec('sec_analysis.tex') + r"""

\section{Conclusion}
""" + load_sec('sec_conclusion.tex') + r"""

""" + (IEEE_DIR / 'table1_main.tex').read_text() + r"""
""" + (IEEE_DIR / 'table2_distance.tex').read_text() + r"""

\begin{figure}[t]
\centering
\includegraphics[width=\columnwidth]{../../results/paper_fig2_main_results.pdf}
\caption{Accuracy by model and coherence dimension. Dashed line = chance (50\%).}
\label{fig:main_results}
\end{figure}

\begin{figure}[t]
\centering
\includegraphics[width=\columnwidth]{../../results/paper_fig3_context_length.pdf}
\caption{Accuracy vs.\ context length. Dashed line = chance (50\%).}
\label{fig:ctx_length}
\end{figure}

\begin{figure}[t]
\centering
\includegraphics[width=\columnwidth]{../../results/paper_fig4_error_analysis.pdf}
\caption{Error breakdown: FP = over-flagged INCONSISTENT; FN = missed inconsistencies.}
\label{fig:errors}
\end{figure}

\bibliographystyle{IEEEtran}
\bibliography{references}

\end{document}
"""

(IEEE_DIR / 'main_ieee.tex').write_text(ieee_tex)
print('✅ Saved: paper/ieee/main_ieee.tex')
print('\nTo compile:')
print('  cd paper/ieee')
print('  pdflatex main_ieee.tex && bibtex main_ieee && pdflatex main_ieee.tex && pdflatex main_ieee.tex')

In [ ]:
# ─── Checklist ────────────────────────────────────────────────────
files = list(IEEE_DIR.glob('*'))
print('IEEE FORMAT — OUTPUT FILES')
print('=' * 45)
for f in sorted(files):
    sz = f.stat().st_size
    print(f'  ✅ {f.name:<30} ({sz:,} bytes)')

print()
print('IEEE-specific compliance notes:')
notes = [
    '\\documentclass[conference]{IEEEtran}   ✅',
    '\\IEEEauthorblockN / \\IEEEauthorblockA  ✅',
    'Both authors with email & affiliation   ✅',
    'IEEEkeywords block                      ✅',
    '\\bibliographystyle{IEEEtran}            ✅',
    'Tables use \\hline (IEEE convention)     ✅',
    'Figures use \\columnwidth               ✅',
    'Page limit check (max 6 pp typical)     ⬜ — verify after compile',
]
for n in notes:
    print(f'  {n}')